# Retail Sales Prediction using Regression Pipeline

## Objective
The goal is to predict the number of items sold at a retail store using historical transaction data.

This notebook focuses on:
- Feature Engineering (Date-based features)
- Time-based Train-Test Split
- Pipeline-based preprocessing
- Regression models

## Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error

## Data Loading

In [ ]:
df = pd.read_csv('../data/q3_retail_promotions.csv')
df.head()

## 1. Date Feature Engineering

Extracting useful time-based features from the transaction_date column.

In [ ]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek

df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

df.head()

New features such as year, month, and day_of_week capture seasonal patterns in sales.  
The is_month_end feature helps identify increased purchasing behavior typically seen toward the end of the month.

## 2. Temporal Train-Test Split

In [ ]:
df = df.sort_values(by='transaction_date')

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

X_train = train_df.drop(['items_sold', 'transaction_date'], axis=1)
y_train = train_df['items_sold']

X_test = test_df.drop(['items_sold', 'transaction_date'], axis=1)
y_test = test_df['items_sold']

A random split is inappropriate for time-series data because it can lead to data leakage, where future information is used to predict past outcomes.  
By using a temporal split, we ensure that the model is trained on past data and evaluated on future data, mimicking real-world scenarios.

## 3. Preprocessing Pipeline

In [ ]:
categorical_cols = ['promotion_type', 'location_type', 'store_size']
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first'), categorical_cols),
    ('num', StandardScaler(), numerical_cols)
])

## 4. Model Training & Evaluation

## Linear Regression Model

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print("Linear Regression RMSE:", rmse_lr)
print("Linear Regression MAE:", mae_lr)

## Random Forest Model

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print("Random Forest RMSE:", rmse_rf)
print("Random Forest MAE:", mae_rf)

## Parity Plot (Predicted vs Actual)

In [ ]:
plt.scatter(y_test, y_pred_rf)
plt.xlabel("Actual Items Sold")
plt.ylabel("Predicted Items Sold")

# Diagonal line
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')

plt.title("Parity Plot - Random Forest")
plt.show()

## Feature Importance

In [ ]:
model = rf_pipeline.named_steps['model']

# Get feature names after preprocessing
feature_names = rf_pipeline.named_steps['preprocessing'].get_feature_names_out()

importances = model.feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

feature_importance_df.head(5)

The most important features influencing sales include promotion-related variables, store characteristics, and temporal factors.  
These insights can help businesses optimize promotions and inventory planning.